# Formatting and cleaning

## 1. Load basic modules and the csv file 

In [1]:
import pandas as pd
import numpy as np
import random as rd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler


data = pd.read_csv('../data_raw/train.csv')
data.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


## 2. Checking Dataset Shape,  null Values and feature types

**Fixing incorrect, incomplete, duplicate or otherwise erroneous data**

In [2]:
data.shape

(381109, 12)

In [3]:
data.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

In [4]:
data.dtypes

id                        int64
Gender                   object
Age                       int64
Driving_License           int64
Region_Code             float64
Previously_Insured        int64
Vehicle_Age              object
Vehicle_Damage           object
Annual_Premium          float64
Policy_Sales_Channel    float64
Vintage                   int64
Response                  int64
dtype: object

**no nulls, both categorical and continues data**
Gender: categorical, balance
Age: continous, left skewed
Driving_License: categorical unbalance (99.78% yes)
Region_Code: categorica, ????
Previously_Insured: categorical 54% no%
Vehicle_Age: categorical, 1 variable more unbalance
Vehicle_Damage: categorical, 51% yes
Annual_Premium: continues, ¿outliers?
Policy_Sales_Channel: agrupar primeros 5, otros agruparlos en "Otros"
Vintage: continues, uniform distribution
Response: target, unbalance, 87% no


## 3. Importing functions from Cleaning_functions.ipynb

In [5]:
import sys
sys.path.append('../src/')
import Cleaning_functions as cf

### 3.1 Continous variables

**Removing outliers from the "Annual_Premium" column**. 

The original data are found with a large scatter, outliers are interpreted as errors in data entry. Given the distribution, it is decided to use log transformation. Log transformation  de-emphasizes outliers and allows us to potentially obtain a bell-shaped distribution. The idea is that taking the log of the data can restore symmetry to the data.

**Data with a high dispersion interpreted as erroneous are eliminated. The dataset is reduced by 64879 rows (17%).**

In [6]:
data = cf.correct_Annual_Premium(data)

In [7]:
data.Annual_Premium.describe()

count    316230.000000
mean      36295.492746
std       12811.756175
min        8739.000000
25%       28488.000000
50%       33985.000000
75%       41336.000000
max      540165.000000
Name: Annual_Premium, dtype: float64

In [8]:
len(data['Policy_Sales_Channel'].unique())

148

In [9]:
data['Policy_Sales_Channel'].value_counts(normalize=True)

152.0    0.386134
26.0     0.213455
124.0    0.196126
160.0    0.051390
122.0    0.027138
           ...   
149.0    0.000003
84.0     0.000003
75.0     0.000003
2.0      0.000003
43.0     0.000003
Name: Policy_Sales_Channel, Length: 148, dtype: float64

In [40]:
porcent = data['Policy_Sales_Channel'].value_counts(normalize=True).loc[lambda x: x > 0.05].index
listporcent = list(porcent)
listporcent

[152.0, 26.0, 124.0, 160.0]

In [45]:
data.shape

(316230, 13)

In [53]:
ds = data[data['Policy_Sales_Channel'].isin(listporcent)]

ds['Policy_Sales_Channel'].value_counts()

152.0    122107
26.0      67501
124.0     62021
160.0     16251
Name: Policy_Sales_Channel, dtype: int64

In [75]:
data.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response,log_premium
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1,10.607921
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0,10.420375
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1,10.553049
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0,10.261826
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0,10.221796


## Dummy variables, train/test and standardization

As we have categorical variables, we need to turn them to dummies variables

In [76]:
categoricals = ["Gender", "Driving_License", "Region_Code", "Previously_Insured", "Vehicle_Age", "Policy_Sales_Channel", 
               ]

enc = OneHotEncoder(drop = "first")
X = data[categoricals]
enc.fit(X)
enc.categories_

[array(['Female', 'Male'], dtype=object),
 array([0, 1]),
 array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
        13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.,
        26., 27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38.,
        39., 40., 41., 42., 43., 44., 45., 46., 47., 48., 49., 50., 51.,
        52.]),
 array([0, 1]),
 array(['1-2 Year', '< 1 Year', '> 2 Years'], dtype=object),
 array([  1.,   2.,   3.,   4.,   6.,   7.,   8.,   9.,  10.,  11.,  12.,
         13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,  22.,  23.,
         24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,  33.,  34.,
         35.,  36.,  37.,  38.,  39.,  40.,  42.,  43.,  44.,  45.,  46.,
         47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,  55.,  56.,  57.,
         58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,  66.,  67.,  68.,
         69.,  70.,  71.,  73.,  74.,  75.,  76.,  78.,  79.,  80.,  81.,
         82.,  83.,  84.,  86.,  87.,  8

In [77]:
dummies = enc.transform(X).toarray()

In [78]:
dummies_df = pd.DataFrame(dummies)

In [82]:
col_names = [categoricals[i] + '_' + enc.categories_[i] for i in range(len(categoricals))] 

UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('<U21'), dtype('<U21')) -> dtype('<U21')